In [1]:
import sys
import os

# Add the parent directory to the path so the package is importable
sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd
from pprint import pprint
from llm_data_quality_assistant.llm_integration import combine_results_1
from llm_data_quality_assistant.pipeline import Pipeline
import jupyter_helper_functions

# Read the dataframes
# LLM cleaned
flight_llm_path = "../analysis/repairs/flight/merged_dataset_gpt_4_1_mini_2025_04_14_50_rows_context_simple_approach_1.csv"
df_llm = pd.read_csv(flight_llm_path)

# Original corrupted
flight_orig_path = (
    "../datasets/parker_datasets/flight/flight_cleaned_corrupted_first1000_int.csv"
)
df_original = pd.read_csv(flight_orig_path)

# Parker cleaned
flight_parker_path = "../analysis/repairs/flight/flight-repair.csv"
df_parker = pd.read_csv(flight_parker_path)
df_parker = df_parker[df_original.columns]


print(df_llm.shape, df_parker.shape, df_original.shape)

# Check if all DataFrames have the same shape
if not (df_llm.shape == df_parker.shape == df_original.shape):
    raise ValueError("Not all DataFrames have the same shape.")

# Ensure all DataFrames have the same order of "composed_key"
if "composed_key" not in df_parker.columns or "composed_key" not in df_llm.columns or "composed_key" not in df_original.columns:
    raise ValueError("All DataFrames must contain a 'composed_key' column.")

# Ensure all DataFrames have the same column order
column_order = df_parker.columns.tolist()
df_llm = df_llm[column_order]
df_original = df_original[column_order]

# df_parker = df_parker.sort_values("composed_key").reset_index(drop=True)
# df_llm = df_llm.sort_values("composed_key").reset_index(drop=True)
# df_original = df_original.sort_values("composed_key").reset_index(drop=True)

if not (
    (df_parker["composed_key"].tolist() == df_llm["composed_key"].tolist() == df_original["composed_key"].tolist())
):
    raise ValueError("The 'composed_key' column does not have the same order in all DataFrames.")

# Calculate the percentage of differing cells between df_parker and df_llm (excluding 'composed_key' column)
diff_mask = (df_parker[column_order].values != df_llm[column_order].values)
num_diff = diff_mask.sum()
total_cells = diff_mask.size
percent_diff = (num_diff / total_cells) * 100

print(f"Percentage of differing cells between df_parker and df_llm (excluding 'eudract_number'): {percent_diff:.2f}%")

# Combine results
combined_df = combine_results_1(df_llm=df_llm, df_parker=df_parker, df_original=df_original)

# Show the result
combined_df.head()

(24606, 5) (24606, 5) (24606, 5)
Percentage of differing cells between df_parker and df_llm (excluding 'eudract_number'): 17.88%


,composed_key,actual_arrival,actual_departure,scheduled_arrival,scheduled_departure
0,2011-12-01 - AA-1007-MIA-PHX,3055.0,2768.0,3065.0,2755.0
1,2011-12-01 - AA-1007-MIA-PHX,3055.0,2768.0,3065.0,2755.0
2,2011-12-01 - AA-1007-MIA-PHX,3055.0,2768.0,3065.0,2755.0
3,2011-12-01 - AA-1007-MIA-PHX,3055.0,2768.0,3065.0,2755.0
4,2011-12-01 - AA-1007-MIA-PHX,3055.0,2768.0,3065.0,2755.0


In [3]:
gold_standard_path = (
    "../datasets/parker_datasets/flight/flight_cleaned_gold_first1000_int.csv"
)
gold_standard = pd.read_csv(gold_standard_path)
# Ensure gold_standard has the same column order and row order as combined_df
gold_standard = gold_standard[column_order]

micro_eval = jupyter_helper_functions.standardize_and_evaluate(
    gold_standard=gold_standard,
    merged_df=combined_df,
    corrupt_dataset=df_original,
    primary_key="composed_key",
    time_delta=0,
    results_dir="../analysis/results/flight",
    file_name="gpt_4_1_mini_50_row_context_option_1"
)


{'accuracy': 0.7690705518979111,
 'column_names': ['composed_key',
                  'actual_arrival',
                  'actual_departure',
                  'scheduled_arrival',
                  'scheduled_departure'],
 'f1_score': 0.76183290895183,
 'false_negative': 11111,
 'false_negative_rate': 0.2340981396034806,
 'false_positive': 11618,
 'false_positive_rate': 0.22797825788347953,
 'num_columns': 5,
 'num_rows': 24606,
 'precision': 0.7578069626850115,
 'recall': 0.7659018603965194,
 'time_taken': 0,
 'true_negative': 39343,
 'true_positive': 36352}
{'column_names': ['composed_key',
                  'actual_arrival',
                  'actual_departure',
                  'scheduled_arrival',
                  'scheduled_departure'],
 'num_columns': 5,
 'num_rows': 24606,
 'stats': [{'accuracy': 0.7370966430951801,
            'column_name': 'actual_arrival',
            'f1_score': 0.751869893751678,
            'false_negative': 2792,
            'false_negative_rate': 0.2